In [1]:
import pandas as pd

df = pd.read_csv("Maharashtra_AllCities_Hospital_Dataset.csv")

print(df.shape)
print(df.head())

(851, 47)
  Hospital_ID     District    City        State City_Type  \
0     MH-1001  Mumbai City  Colaba  Maharashtra     Urban   
1     MH-1002  Mumbai City  Colaba  Maharashtra     Urban   
2     MH-1003  Mumbai City  Colaba  Maharashtra     Urban   
3     MH-1004  Mumbai City  Colaba  Maharashtra     Urban   
4     MH-1005  Mumbai City    Fort  Maharashtra     Urban   

                   Hospital_Name Reference_Period_AU595  Total_Beds  ICU_Beds  \
0        City Health Care Colaba             31.08.2020         119        33   
1  Central Hospital & ICU Colaba             31.08.2020         133        32   
2         Sparsh Hospital Colaba             31.08.2020         137        52   
3    Fortis Trauma Centre Colaba             31.08.2020          94        41   
4   Multispecialty Hospital Fort             31.08.2020         119        31   

   Non_ICU_Beds  ...  Doctors_Orthopedic  Doctors_Pediatrician  Total_Doctors  \
0            86  ...                   2               

In [2]:
import numpy as np

np.random.seed(42)

# Simulate patient load (30% to 95% of beds)
df['patients'] = (
    df['Total_Beds'] * np.random.uniform(0.3, 0.95, size=len(df))
).astype(int)

print(df[['Total_Beds', 'patients']].head())

   Total_Beds  patients
0         119        64
1         133       122
2         137       106
3          94        64
4         119        47


In [3]:
import numpy as np

np.random.seed(42)

df['doctors_needed'] = (
    (df['patients'] / 10) +
    np.random.normal(0, 1, size=len(df))   # noise
).round().clip(lower=1)

df['nurses_needed'] = (
    (df['patients'] / 5) +
    np.random.normal(0, 2, size=len(df))
).round().clip(lower=1)

df['surgeons_needed'] = (
    (df['patients'] / 25) +
    np.random.normal(0, 1, size=len(df))
).round().clip(lower=1)

In [4]:
df['urban_pressure'] = (df['City_Type'] == 'Urban').astype(int)

df['equipment_strength'] = (
    df['Ventilators'] +
    df['Oxygen_Concentrators']
) / df['Total_Beds'].replace(0, 1)

In [5]:
df['patient_doctor_pressure'] = df['patients'] / df['Total_Doctors'].replace(0, 1)

In [6]:
df['utilization'] = df['patients'] / df['Total_Beds'].replace(0, 1)

df['icu_ratio'] = df['ICU_Beds'] / df['Total_Beds'].replace(0, 1)

df['doctor_bed_ratio'] = df['Total_Doctors'] / df['Total_Beds'].replace(0, 1)

df = df.fillna(0)

In [7]:
df['patients'] = (
    df['Total_Beds'] *
    np.random.uniform(0.3, 0.95, size=len(df)) *
    np.random.uniform(0.9, 1.1, size=len(df))  # extra variation
).astype(int)

In [8]:
from sklearn.preprocessing import LabelEncoder

le_city = LabelEncoder()
le_district = LabelEncoder()

df['City_enc'] = le_city.fit_transform(df['City'])
df['District_enc'] = le_district.fit_transform(df['District'])

In [9]:
FEATURES = [
    'Total_Beds',
    'utilization',
    'icu_ratio',
    'doctor_bed_ratio',
    'City_enc',
    'District_enc'
]

X = df[FEATURES]

y_doctors = df['doctors_needed']
y_nurses = df['nurses_needed']
y_surgeons = df['surgeons_needed']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, yd_train, yd_test = train_test_split(X, y_doctors, test_size=0.2, random_state=42)
_, _, yn_train, yn_test = train_test_split(X, y_nurses, test_size=0.2, random_state=42)
_, _, ys_train, ys_test = train_test_split(X, y_surgeons, test_size=0.2, random_state=42)

In [11]:
from sklearn.ensemble import GradientBoostingRegressor

model_doc = GradientBoostingRegressor().fit(X_train, yd_train)
model_nurse = GradientBoostingRegressor().fit(X_train, yn_train)
model_surg = GradientBoostingRegressor().fit(X_train, ys_train)

print("All models trained!")

All models trained!


In [12]:
from sklearn.metrics import mean_absolute_error

print("Doctors MAE:", mean_absolute_error(yd_test, model_doc.predict(X_test)))
print("Nurses MAE:", mean_absolute_error(yn_test, model_nurse.predict(X_test)))
print("Surgeons MAE:", mean_absolute_error(ys_test, model_surg.predict(X_test)))

Doctors MAE: 0.5487624083962492
Nurses MAE: 1.4572337102735993
Surgeons MAE: 0.36256834341530125


In [13]:
sample = X_test.iloc[0:1]

print("Predicted Doctors:", model_doc.predict(sample))
print("Predicted Nurses:", model_nurse.predict(sample))
print("Predicted Surgeons:", model_surg.predict(sample))

Predicted Doctors: [2.85045241]
Predicted Nurses: [4.79110497]
Predicted Surgeons: [1.11882304]


In [14]:
def hospital_pipeline(input_data):
    import pandas as pd
    
    df = pd.DataFrame([input_data])
    
    df['icu_ratio'] = df['ICU_Beds'] / df['Total_Beds']
    df['nonicu_ratio'] = df['Non_ICU_Beds'] / df['Total_Beds']
    df['doctor_bed_ratio'] = df['Total_Doctors'] / df['Total_Beds']
    df['ventilator_ratio'] = df['Ventilators'] / df['Total_Beds']
    df['room_density'] = df['Total_Rooms'] / df['Total_Beds']
    df['emergency_ratio'] = df['Emergency_Rooms'] / df['Total_Rooms']
    
    df = df.fillna(0)
    
    df['City_enc'] = le_city.transform(df['City'])
    df['District_enc'] = le_district.transform(df['District'])
    df['State_enc'] = le_state.transform(df['State'])
    df['City_Type_enc'] = le_city_type.transform(df['City_Type'])
    
    X_reg_input = df[[
        'Total_Beds', 'Available_Beds',
        'ICU_Beds', 'Non_ICU_Beds',
        'Total_Doctors', 'Ventilators',
        'icu_ratio', 'nonicu_ratio',
        'doctor_bed_ratio', 'ventilator_ratio',
        'room_density', 'emergency_ratio',
        'City_enc', 'District_enc',
        'State_enc', 'City_Type_enc'
    ]]
    
    predicted_beds = reg.predict(X_reg_input)[0]

    utilization = predicted_beds / df['Total_Beds'].values[0]

    df['utilization'] = utilization
    
    X_staff_input = df[[
        'Total_Beds',
        'utilization',
        'icu_ratio',
        'doctor_bed_ratio',
        'City_enc',
        'District_enc'
    ]]
    
    doctors = model_doc.predict(X_staff_input)[0]
    nurses = model_nurse.predict(X_staff_input)[0]
    surgeons = model_surg.predict(X_staff_input)[0]

    X_clf_input = df[[
        'Total_Beds', 'Available_Beds',
        'ICU_Beds', 'Non_ICU_Beds',
        'Total_Doctors', 'Ventilators',
        'icu_ratio', 'nonicu_ratio',
        'doctor_bed_ratio', 'ventilator_ratio',
        'room_density', 'emergency_ratio',
        'City_enc', 'District_enc',
        'State_enc', 'City_Type_enc'
    ]]
    
    surge = clf.predict(X_clf_input)[0]

    return {
        "Beds Occupied": round(predicted_beds),
        "Utilization (%)": round(utilization * 100, 2),
        "Doctors Needed": round(doctors),
        "Nurses Needed": round(nurses),
        "Surgeons Needed": round(surgeons),
        "Surge Status": "YES" if surge == 1 else "NO"
    }

In [16]:
sample = {
    "Total_Beds": 200,
    "Available_Beds": 60,
    "ICU_Beds": 40,
    "Non_ICU_Beds": 160,
    "Total_Doctors": 25,
    "Ventilators": 20,
    "Total_Rooms": 120,
    "Emergency_Rooms": 12,
    
    "City": "Colaba",
    "District": "Mumbai City",
    "State": "Maharashtra",
    "City_Type": "Urban"
}

result = hospital_pipeline(sample)

for k, v in result.items():
    print(k, ":", v)

NameError: name 'le_state' is not defined